# FinHybrid - Phase 3C: Few-Shot Prompt

**Optimization:** Phase 3C-2 - Few-shot examples

**Dataset:** FinHybrid (Financial Reports)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Exact Match ±1%

**Documents:** 4 example PDFs (ADI, ABMD, GS, JKHY)

**What changed:**
- ✅ Few-shot prompt (3 example Q&A pairs to guide model)
- ✅ Keep all Phase 2 parameters (TOP_K=10, CHUNK_SIZE=1500)

**Baseline (Phase 2):**
- Empty rate: 36.2% (17/47 questions)
- Exact Match: 34.04%

**Target:**
- Empty rate: <30% (+3-7 questions)
- Better answer quality with examples

**Expected runtime:** 20-30 minutes
**Expected cost:** +20% tokens (longer prompt)

## Setup and Imports

In [1]:
import sys
import os

# Navigate to project root (4 levels up from 3_prompts/notebooks/)
project_root = os.path.abspath('../../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


In [2]:
import pandas as pd
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess
from uda.utils.prompts import get_prompt  # NEW: Import prompt module
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

✓ All imports successful


/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configuration

In [3]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [4]:
# Experiment Parameters (SAME AS PHASE 2 - only prompt changes)
DATASET_NAME = "fin"
CHUNK_SIZE = 1500  # From Phase 2
CHUNK_OVERLAP = 150
TOP_K = 10  # From Phase 2
TEMPERATURE = 0.1
MAX_TOKENS = 512

# NEW: Prompt type
PROMPT_TYPE = "fewshot"  # instruction, fewshot, or cot

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/finhybrid_fewshot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Prompt type: {PROMPT_TYPE}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: fin
Chunk size: 1500
Top-K: 10
Prompt type: fewshot
Output dir: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/finhybrid_fewshot


## Initialize Models

In [5]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model (local, free)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

# NEW: Load prompt function
prompt_fn = get_prompt(PROMPT_TYPE)
print(f"✓ Prompt function loaded: {PROMPT_TYPE}")

✓ Together AI client initialized


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model loaded: all-MiniLM-L6-v2
✓ Text splitter initialized
✓ Prompt function loaded: fewshot


## Helper Functions

In [6]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF using PyPDF2 (Phase 2 method)"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()
    
    # Delete if exists
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    # Create collection
    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    # Add documents
    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)
    
    return collection

def answer_question(collection, question):
    """
    Retrieve context and generate answer.
    
    CHANGED: Uses few-shot prompt with examples
    """
    # Retrieve (same as Phase 2)
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])
    
    # NEW: Build prompt using prompts module
    prompt_text = prompt_fn(context=context, question=question)
    
    # Convert to message format for Together AI
    messages = [
        {"role": "user", "content": prompt_text}
    ]
    
    # Generate (same as Phase 2)
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    
    return response.choices[0].message.content

print("✓ Helper functions defined")

✓ Helper functions defined


## Load Q&A Data

In [7]:
# Load FinHybrid Q&A
csv_file = "./dataset/qa/fin_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# Filter to only documents with available PDFs
AVAILABLE_DOCS = [
    "ABMD_2012",
    "ADI_2009",
    "GS_2016",
    "JKHY_2015"
]

qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")

Total documents in CSV: 788
Available PDFs: 4

Filtered to documents with PDFs:

  ABMD_2012: 12 Q&A pairs
  ADI_2009: 9 Q&A pairs
  GS_2016: 23 Q&A pairs
  JKHY_2015: 3 Q&A pairs

Total Q&A to process: 47


## Main Processing Loop

**This will process 4 documents with 47 Q&A pairs**

**Expected runtime:** 20-30 minutes

In [8]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")
    
    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue
    
    print(f"PDF: {pdf_path}")
    
    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")
    
    # Build index
    print("Building vector index...")
    collection = build_index(text_chunks, collection_name=f"fin_{doc_name}_prompt")
    print("✓ Index built")
    
    # Process each question
    print(f"\nAnswering {len(doc_qas)} questions...")
    
    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")
        
        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")
            
            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
                "prompt_type": PROMPT_TYPE,
            })
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print(f"\n✓ Completed {doc_name}: {len([r for r in all_results if r['doc'] == doc_name])} questions processed")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")


Processing: ADI_2009
PDF: dataset/src_doc_files_example/fin_docs/ADI_2009.pdf
Extracting text...
Created 284 chunks
Building vector index...
✓ Index built

Answering 9 questions...

[1/9] what is the the interest expense in 2009?...
   Answer: ...

[2/9] what is the expected growth rate in amortization expense in 2010?...
   Answer: ...

[3/9] what is the net difference between in amounts used to as hedging instr...
   Answer: ...

[4/9] what is the growth rate in amortization expense in 2009?...
   Answer: ...

[5/9] what is the net change in the balance of total amounts of uncertain ta...
   Answer: ...

[6/9] what is the percentage increase in interest expanse and penalties in 2...
   Answer: ...

[7/9] what is the lobor rate as of october 31 , 2009?...
   Answer: ...

[8/9] what percentage did the balance increase from 2007 to 2009?...
   Answer: The balance increased from $9,889 thousand (November 3, 2007) to $18,161 thousan...

[9/9] what would be the balance if the company suff

## Diagnostic: Check Empty Responses

In [9]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)
    
    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")
    
    # COMPARISON WITH PHASE 2
    phase2_empty = 17
    phase2_total = 47
    phase2_empty_pct = phase2_empty / phase2_total * 100
    
    improvement = phase2_empty - empty_count
    improvement_pct = phase2_empty_pct - (empty_count/total_count*100)
    
    print(f"\n{'='*80}")
    print(f"COMPARISON WITH PHASE 2 BASELINE")
    print(f"{'='*80}")
    print(f"Phase 2 (Baseline): {phase2_empty}/{phase2_total} empty ({phase2_empty_pct:.1f}%)")
    print(f"Phase 3C (Few-Shot): {empty_count}/{total_count} empty ({empty_count/total_count*100:.1f}%)")
    print(f"\nImprovement: {improvement:+d} questions ({improvement_pct:+.1f} percentage points)")
    
    if improvement > 0:
        print(f"✅ SUCCESS: Few-shot prompts reduced empty responses!")
    elif improvement == 0:
        print(f"⚠️  NEUTRAL: No change in empty responses")
    else:
        print(f"❌ REGRESSION: Empty responses increased")
    
    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
else:
    print("❌ No results to analyze")


DIAGNOSTIC: Empty Response Analysis
Total Q&A processed: 47
Empty responses: 19 (40.4%)
Answered: 28 (59.6%)

COMPARISON WITH PHASE 2 BASELINE
Phase 2 (Baseline): 17/47 empty (36.2%)
Phase 3C (Few-Shot): 19/47 empty (40.4%)

Improvement: -2 questions (-4.3 percentage points)
❌ REGRESSION: Empty responses increased

Empty responses by document:
  ADI_2009: 8/9 empty (88.9%)
  ABMD_2012: 4/12 empty (33.3%)
  GS_2016: 6/23 empty (26.1%)
  JKHY_2015: 1/3 empty (33.3%)


## Evaluate Results

In [10]:
if all_results:
    print("\nEvaluating FinHybrid results (Exact Match ±1%)...")
    eval_main(DATASET_NAME, all_results)
else:
    print("❌ No results to evaluate")


Evaluating FinHybrid results (Exact Match ±1%)...
Exact-match accuracy: 19.15


## Save Results

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"finhybrid_fewshot_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")
    
    # Summary by document
    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/finhybrid_fewshot/finhybrid_fewshot_20260629_221357.csv
Total Q&A: 47

Results by document:
  ADI_2009: 9 questions
  ABMD_2012: 12 questions
  GS_2016: 23 questions
  JKHY_2015: 3 questions


## Final Summary

In [12]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    empty_count = results_df['response'].fillna('').str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count
    
    phase2_empty = 17
    improvement = phase2_empty - empty_count
    
    print(f"\n{'='*80}")
    print(f"FINAL SUMMARY - FEW-SHOT PROMPT")
    print(f"{'='*80}")
    print(f"Dataset: FinHybrid (47 Q&A)")
    print(f"Prompt type: {PROMPT_TYPE}")
    print(f"\nResults:")
    print(f"  Answered: {answered_count}/{len(results_df)} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"  Empty: {empty_count}/{len(results_df)} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"\nVs Phase 2 Baseline:")
    print(f"  Change: {improvement:+d} questions")
    print(f"  Expected: +3 to +7 questions")
    
    if improvement >= 3:
        print(f"\n✅ SUCCESS: Met or exceeded expectations!")
        print(f"   Next step: Test CoT variant or scale to all datasets")
    elif improvement > 0:
        print(f"\n⚠️  PARTIAL: Some improvement but below expectations")
        print(f"   Next step: Try CoT variant")
    else:
        print(f"\n❌ FAILED: No improvement or regression")
        print(f"   Next step: Investigate why, may revert to instruction")
else:
    print("\n❌ No results to summarize")


FINAL SUMMARY - FEW-SHOT PROMPT
Dataset: FinHybrid (47 Q&A)
Prompt type: fewshot

Results:
  Answered: 28/47 (59.6%)
  Empty: 19/47 (40.4%)

Vs Phase 2 Baseline:
  Change: -2 questions
  Expected: +3 to +7 questions

❌ FAILED: No improvement or regression
   Next step: Investigate why, may revert to instruction


---

## Done!

**Results saved to:** `./results/finhybrid_fewshot/`

**Next steps:**
1. Compare with instruction prompt results
2. If few-shot better: Test CoT or scale to all datasets
3. If instruction better: Use instruction for all datasets